# 🕉️ Sadhana Nandadeep — Data Pipeline

This notebook is the **single entry point** for processing new videos end-to-end:
1. Download YouTube transcripts → Google Drive
2. Extract metadata with Gemini AI
3. Upload metadata to DynamoDB
4. Generate embeddings on T4 GPU
5. Upload vectors to Qdrant Cloud (Hybrid Search)

> **Runtime**: Change to **T4 GPU** via `Runtime → Change runtime type`.
> **Secrets**: Add all API keys in Colab's 🔑 Secrets sidebar before running.

---

## Part 1: Environment Setup

These cells prepare the Colab environment. Run them once at the start of every session.

### Cell 1 — Mount Google Drive & Create Folders

Mounts Google Drive and creates the permanent storage directories.
All pipeline data (transcripts, metadata, embeddings) is stored on Drive, so nothing is lost when the Colab runtime disconnects.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — safe to re-run. `os.makedirs` with `exist_ok=True` won't overwrite anything. |
| 📁 **Creates** | `SadhanaNandadeep_Data/{output, enriched_metadata, enriched_json}` on Google Drive |

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the permanent Drive folder structure
DRIVE_ROOT = '/content/drive/MyDrive/SadhanaNandadeep_Data'
DRIVE_OUTPUT = os.path.join(DRIVE_ROOT, 'output')
DRIVE_META = os.path.join(DRIVE_ROOT, 'enriched_metadata')
DRIVE_JSON = os.path.join(DRIVE_ROOT, 'enriched_json')

# 3. Create folders if they don't exist
for d in [DRIVE_ROOT, DRIVE_OUTPUT, DRIVE_META, DRIVE_JSON]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Google Drive mounted! Data will be stored in: {DRIVE_ROOT}")


### Cell 2 — Install Dependencies & Load API Keys

Installs all required Python packages and loads API credentials from Colab Secrets.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — `pip install -q` skips already-installed packages. |
| 🔑 **Required Secrets** | `GEMINI_API_KEY`, `QDRANT_URL`, `QDRANT_API_KEY`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION` |

In [ ]:
!pip install -q \\
    "google-genai>=1.66.0,<2.0.0" \\
    langchain==1.3.1 \\
    langchain-text-splitters==1.1.2 \\
    langchain-core==1.4.0 \\
    sentence-transformers \\
    torch \\
    qdrant-client==1.18.0 \\
    youtube-transcript-api \\
    python-dotenv \\
    tenacity \\
    boto3

import os
from google.colab import userdata

# Load secrets into environment (ensure these are added in Colab's 🔑 sidebar)
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["QDRANT_URL"]     = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = userdata.get("AWS_DEFAULT_REGION")

print("✅ Dependencies installed and secrets loaded")


### Cell 3 — Clone Repository & Create Symlinks

Clones the latest code from GitHub and creates **symbolic links** so the code reads/writes directly to Google Drive.
This means all data is persisted even if the Colab runtime disconnects.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — deletes the old clone and re-clones fresh every time. Data on Drive is never touched. |
| 🔗 **Symlinks** | `data_pipeline/{output, enriched_metadata, enriched_json}` → Google Drive folders |
| 🔑 **Required Secret** | `GITHUB_TOKEN` (for private repo access) |

In [ ]:
import os
from google.colab import userdata

%cd /content
!rm -rf /content/repo

# Clone Repo using the token from the Colab kernel
token = userdata.get("GITHUB_TOKEN")
!git clone https://{token}@github.com/ameyk2004/multimodal-video-search.git /content/repo --quiet

# Create symlinks so the code writes directly to Google Drive
!rm -rf /content/repo/data_pipeline/output
!rm -rf /content/repo/data_pipeline/enriched_metadata
!rm -rf /content/repo/data_pipeline/enriched_json

!ln -s /content/drive/MyDrive/SadhanaNandadeep_Data/output /content/repo/data_pipeline/output
!ln -s /content/drive/MyDrive/SadhanaNandadeep_Data/enriched_metadata /content/repo/data_pipeline/enriched_metadata
!ln -s /content/drive/MyDrive/SadhanaNandadeep_Data/enriched_json /content/repo/data_pipeline/enriched_json

# Install package
!pip install -q -e /content/repo

print("✅ Repo cloned and Google Drive symlinks created.")


---
## Part 2: Data Extraction & Enrichment (CPU)

These cells fetch transcripts, extract metadata, and upload to DynamoDB. No GPU required.

### Cell 4 — Fetch Raw YouTube Transcripts

Downloads fine-grained (2-5 second) Marathi transcripts for the video URLs defined in `data_pipeline/main.py`.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — **skips** any video whose transcript JSON already exists in `output/`. |
| ⚠️ **IP Block Risk** | YouTube sometimes blocks Colab's cloud IPs. If you see `IpBlocked`, try resetting the runtime or running this step locally on your Mac instead. |
| 📄 **Output** | One `.json` file per video in `data_pipeline/output/` |

In [ ]:
%cd /content/repo
!python data_pipeline/main.py


### Cell 5 — Fix Timestamps on Existing Metadata (Optional)

Re-resolves `start_time_seconds` for stories and musical segments using the raw transcripts, **without** re-calling the Gemini API.

Only needed if you've re-downloaded raw transcripts and want to patch timestamps in already-existing metadata.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — only updates items where the timestamp changed by more than 0.5 seconds. |
| 🧠 **Gemini calls?** | ❌ None — purely local computation. |
| 🔧 **Dry run** | Add `--dry-run` flag to preview changes without writing files. |

In [ ]:
%cd /content/repo
!python scripts/metadata/fix_timestamps.py


### Cell 6 — AI Video Enrichment (Gemini)

Sends each video's full transcript to Gemini to extract structured metadata: topics, queries, stories, actionable practices, quoted verses, and musical segments.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — **skips** any video whose `_meta.json` already exists in `enriched_metadata/`. |
| ⏱️ **Duration** | ~5-10 seconds per new video (Gemini API + rate limiting delay). |
| 💰 **Cost** | Uses Gemini API quota. Already-processed videos are free (skipped). |
| 📄 **Output** | One `<video_id>_meta.json` per video in `data_pipeline/enriched_metadata/` |

In [ ]:
%cd /content/repo
!python -m data_pipeline.video_enricher


### Cell 7 — Upload Metadata to DynamoDB

Uploads the enriched metadata (topics, queries, stories, musical segments) to the AWS DynamoDB table.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ⚠️ **Upsert behavior** — existing records with the same `video_id` are **overwritten** with the latest metadata. New records are inserted. No data is deleted. |
| ☁️ **Target** | DynamoDB table `sadhananandadeep-content` |

In [ ]:
%cd /content/repo
!python data_pipeline/dynamo_uploader.py


---
## Part 3: GPU Embedding & Qdrant Upload

These cells require the **T4 GPU runtime**. They generate dense embeddings and upload vectors to Qdrant Cloud.

### Cell 8 — Load Embedding Model (BGE-M3 on GPU)

Loads the `BAAI/bge-m3` sentence transformer model onto the GPU. This is a large model (~1 GB download on first run, ~1 minute to load).

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ Yes — just loads the model into memory. No data changes. |
| 🖥️ **Requires** | T4 GPU runtime (falls back to CPU if unavailable, but will be very slow). |

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
embedder = SentenceTransformer("BAAI/bge-m3", device=device)
print("✅ Embedder ready")


### Cell 9 — Embed Queries & Video Chunks

This cell does **two things**:

1. **Embeds search queries** from DynamoDB and uploads them to the `sadhananandadeep-queries` Qdrant collection.
2. **Embeds video transcript chunks** using the GPU and saves the result as `_enriched.json` files.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ✅ **Partially** |
| | — **Queries**: Appends only new queries. Existing ones are skipped (checked by deterministic UUID). The query collection is **never deleted**. |
| | — **Chunks**: Skips any video whose `_enriched.json` already exists. |
| 🔑 **Required Secret** | `HF_API_KEY` (HuggingFace, for query embedding via API) |
| 📄 **Output** | `data_pipeline/enriched_json/<video_id>_enriched.json` per video |

In [ ]:
import os
from google.colab import userdata

# ── Step 1: Embed and upload search queries ──
os.environ["HF_API_KEY"] = userdata.get("HF_API_KEY")
!cd /content/repo && python scripts/qdrant/embed_and_upload_queries.py --yes

# ── Step 2: Embed video transcript chunks on GPU ──
import sys
sys.path.append('/content/repo')

import json, glob
from data_pipeline.transcript_processor import TranscriptProcessor

processor = TranscriptProcessor()
raw_files = sorted(glob.glob("/content/repo/data_pipeline/output/*.json"))
total_files = len(raw_files)
print(f"\n🔢 Found {total_files} raw transcript files to process.")

skipped = 0
processed = 0

for idx, filepath in enumerate(raw_files, start=1):
    video_id = os.path.splitext(os.path.basename(filepath))[0]
    out_path = f"/content/repo/data_pipeline/enriched_json/{video_id}_enriched.json"

    if os.path.exists(out_path):
        skipped += 1
        continue

    print(f"[{idx}/{total_files}] 🔄 Embedding chunks for {video_id}...")
    chunks, _, _ = processor.process_file(filepath, video_id)

    for chunk in chunks:
        chunk["embedding_vector"] = embedder.encode(chunk["marathi_raw"]).tolist()

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    processed += 1
    print(f"    ✅ Saved {len(chunks)} chunks")

print(f"\n{"="*50}")
print(f"📊 EMBEDDING SUMMARY")
print(f"{"="*50}")
print(f"Total raw files:     {total_files}")
print(f"Newly embedded:      {processed}")
print(f"Skipped (existing):  {skipped}")
print(f"{"="*50}")
print("✅ Query and Chunk Embedding complete!")


### Cell 10 — Upload to Qdrant (Hybrid Search)

Builds BM25 sparse vocabulary, computes sparse vectors, and uploads **all** video chunks (dense + sparse) to the Qdrant `sadhananandadeep-videos` collection.

| Property | Value |
|----------|-------|
| ⚡ **Idempotent?** | ⚠️ **NO — DESTRUCTIVE** |
| | This cell **DELETES the entire `sadhananandadeep-videos` collection** and rebuilds it from scratch using all `_enriched.json` files. |
| | However, all your old video data is safe in Google Drive — they get re-uploaded along with new ones. |
| | The **query collection (`sadhananandadeep-queries`) is NOT affected**. |
| 📄 **Also saves** | `cloud-backend/lambdas/similarity_search/vocab_idf.json` (BM25 vocabulary for Lambda) |

In [ ]:
%cd /content/repo
!python scripts/qdrant/rebuild_hybrid_collection.py


---
## Part 4: Verification

Quick sanity checks to confirm the pipeline ran correctly.

### Cell 11 — Verify Pipeline Output

Counts the files in each output directory to confirm everything was processed.

In [ ]:
import os

dirs = {
    "Raw Transcripts (output/)": "/content/repo/data_pipeline/output",
    "Enriched Metadata (enriched_metadata/)": "/content/repo/data_pipeline/enriched_metadata",
    "Embedded Chunks (enriched_json/)": "/content/repo/data_pipeline/enriched_json",
}

print("=" * 50)
print("📊 PIPELINE OUTPUT VERIFICATION")
print("=" * 50)

for label, path in dirs.items():
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
        print(f"{label}: {len(files)} files")
    else:
        print(f"{label}: ❌ Directory not found")

print("=" * 50)
print("\n🎉 Pipeline verification complete!")


---
*Pipeline notebook — last updated June 2026.*